# Exercise 1 

In this exercise, we will work with the `spam.csv` data file. This file contains text and a label attached to each text. The goal is to clean the text data and then build classifcation models. 

### Exercise 1(a) (2 points)

Load the `pandas` and `numpy` libraries.

In [10]:
import pandas as pd
import numpy as np 

### Exercise 1(b) (2 points)

Read the `csv` file and create a data-frame called `spam_df`.

In [11]:
spam_df = pd.read_csv('DATA/spam.csv')
spam_df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Exercise 1(c) (2 points)

Report the frequency table of `label`.

In [12]:
spam_df['label'].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

### Exercise 1(d) (2 points)

Using the `string` library, remove all the punctuation of the `text` column.

In [13]:
import string, re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

nltk.download(['stopwords', 'punkt_tab', 'wordnet',
               'averaged_perceptron_tagger_eng'])

spam_df['text_clean'] = spam_df['text'].str.translate(
    str.maketrans('', '', string.punctuation)
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


### Exercise 1(e) (10 points)

Using the `nltk` library, remove stopwords and pronouns from `text_clean` and store in another column called `'text_clean_1'`. 

In [14]:
stop = set(stopwords.words('english'))
def drop_stops_pronouns(t):
    toks = word_tokenize(str(t).lower())
    tagged = pos_tag(toks)
    return ' '.join(w for w, tag in tagged
                    if w not in stop and tag not in ('PRP','PRP$','WP','WP$'))
spam_df['text_clean_1'] = spam_df['text_clean'].apply(drop_stops_pronouns)

### Exercise 1(f) (10 points)

Perform Lemmatization of `text_clean_1` and store the results in another column called `text_clean_2` in the `spam_df` data frame.

In [15]:
lem = WordNetLemmatizer()
spam_df['text_clean_2'] = spam_df['text_clean_1'].apply(
    lambda t: ' '.join(lem.lemmatize(w) for w in t.split())
)

### Exercise 1(g) (5 points)

Based on the results from part 1(f), we see digits in some of the text in `text_clean_2`. Remove all the digits from `text_clean_2` and store the results in another column `text_clean_3` in the `spam_df` data frame.

In [16]:
spam_df['text_clean_3'] = spam_df['text_clean_2'].str.replace(r'\d+', '', regex=True).str.strip()

### Exercise 1(h) (5 points)

Using the `CounterVectorizer` function from `sklearn.feature_extraction.text`, convert the `text_clean_3` column into matrix of token counts. Use `max_features=30` to create the input features, and map `label` to 0-1.

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=30)
X = cv.fit_transform(spam_df['text_clean_3'])
y = spam_df['label'].map({'ham': 0, 'spam': 1})

### Exercise 1(i) (7 points)

Using 5-fold cross-validation strategy (use `StratifiedKFold`), train and evaluate the out-of-sample performance of the `LogisticRegression` model. Use `accuracy_score` as measure of performance.

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logit_scores = cross_val_score(LogisticRegression(max_iter=1000), X, y,
                               cv=skf, scoring='accuracy')
print(logit_scores, logit_scores.mean())

[0.93004484 0.94529148 0.93536804 0.93806104 0.9443447 ] 0.9386220222041526


### Exercise 1(j) (7 points)

Using the same cross-validation strategy from part 1(i), train and evaluate the out-of-fold performance of the `MultinomialNB` model. Use `accuracy_score` as measure of performance. 

In [21]:
from sklearn.naive_bayes import MultinomialNB

nb_scores = cross_val_score(MultinomialNB(), X, y, cv=skf, scoring='accuracy')
print(nb_scores, nb_scores.mean())

[0.90941704 0.92556054 0.91651706 0.9048474  0.91921005] 0.9151104169517998


### Exercise 1(k) (3 points)

Based on the results from parts 1(i) and 1(j), what model would use to predict spam label?

Logistic regression averaged 93.9% accuracy across the five folds compared to 91.5% for multinomial naive Bayes, and it won in every individual fold, so I would use logistic regression to predict the spam label.